# 🏛️ Auditoría Analítica SECOP II - Pipeline de Ingesta y Modelo NoSQL
### Proyecto Final de Big Data & Data Lakehouse

---

## 👥 1. Integrantes del Equipo
* **Miguel Andres Salas Castro**
* **Zuly Natalia Cely Barbosa **
* **Nino Javier Valoyes Morales**

---

## 📅 2. Control de Trazabilidad e Integridad (Data Lineage)
* **Fecha y Hora de Cierre de Descarga:** 3 de junio de 2026 - 09:00 PM
* **Universo de Registros Procesados:** `1,687,554` filas consolidadas.

---

## 📐 3. Decisiones de Arquitectura
El ecosistema se diseñó bajo una arquitectura **Data Lakehouse** distribuida utilizando **PySpark (Spark SQL)** sobre nodos **Databricks Serverless**, garantizando un procesamiento eficiente en memoria para los más de 1.6 millones de transacciones. 

El pipeline está estrictamente dividido en dos componentes lógicos independientes (Notebooks):

📂 taller_final/
├── 📓 Taller_final_descarga.ipynb  (Capa Bronze / Ingesta Resiliente)

└── 📓 Analisis.ipynb  (Capa Silver y Gold / Modelo NoSQL)

Datalake:
/Volumes/workspace/default/taller_final/raw/adiciones
/Volumes/workspace/default/taller_final/raw/contratos
/Volumes/workspace/default/taller_final/raw/divipola
/Volumes/workspace/default/taller_final/raw/ejecucion


### Componente 1: Taller_final_descarga (`Notebook 1`)
1. **Auditoría Previa:** Realiza consultas directas a los endpoints mediante `count(*)` para verificar el universo total de registros disponibles en la API de Socrata de los años objetivo (**2025** y **2026**).
2. **Estrategia de Chunking:** Descarga de datos de forma paginada mediante bloques masivos de **50,000 registros** a través de parámetros límites (`$limit`) y desplazamientos (`$offset`).
3. **Mecanismo de Checkpoints (Puntos de Control):** Antes de iniciar cada iteración, el script escanea el almacenamiento físico. Si detecta una caída de red o desconexión del clúster, el pipeline **reanuda automáticamente** en el último bloque guardado con éxito, protegiendo la consistencia de los archivos binarios **Parquet** de la capa Bronze.

### Componente 2: Taller_final_analisis (`Notebook 2`)
Encargado de la transformación, limpieza avanzada de datos huérfanos y la estructuración del modelo documental NoSQL.

---

## 🏗️ 4. Modelo Documental NoSQL (Colecciones Exportadas)
Tras enriquecer y cruzar los datos de contratos con la tabla geoespacial **DIVIPOLA (DANE)**, la data se transformó en estructuras semiestructuradas (BSON/JSON) optimizadas para **MongoDB Atlas** para alimentar la capa del Dashboard analítico.

Antes de su generación, el pipeline realiza un proceso automático de **Force Purge** (vaciado absoluto del directorio raíz) para garantizar la frescura de los datos. Las 6 colecciones estructuradas son:

1. 📦 **`contratos_operativos`**: Contratos con metadatos y subdocumentos anidados de `detalles_financieros` y `ubicacion_geografica` mediante funciones `F.struct` para evitar JOINs en caliente. Se realizo filtro por prioridad Critica, debido a que la capa gratuita de Mongo permite hata 512 MB. Cunata con 125111 contratos.

![](/Volumes/workspace/default/taller_final/raw/Imagenes/contratos.png)

2. 🚨 **`alertas_revision`**: Colección exclusiva de contratos catalogados con prioridad **ALTA** que contiene el desglose analítico de su matriz de puntos de riesgo.

![](/Volumes/workspace/default/taller_final/raw/Imagenes/alertas.png)

3. 🏛️ **`entidades_resumen`**: Colección pre-agregada por Entidad que almacena presupuestos total contratos, presupuesto y scores de riesgo promedio (*Pattern Computed*).

![](/Volumes/workspace/default/taller_final/raw/Imagenes/Entidades.png)

4. 🤝 **`proveedores_resumen`**: Agregación analítica agrupada por proveedor para evaluar evaluar numero de adiciones y contratos ganados.

![](/Volumes/workspace/default/taller_final/raw/Imagenes/proveedor.png)

5. 🏷️ **`temas_resumen`**: Clúster de contratos consolidados por temáticas críticas de interés fiscal (ej. Tecnología e Informática, Servicios).

![](/Volumes/workspace/default/taller_final/raw/Imagenes/Temas.png)

6. ⚙️ **`metadata_pipeline`**: Documento de auditoría técnica que certifica el éxito del proceso (`SUCCESS`), fecha de ejecución (`2026-06-05`) y número 
total de filas inyectadas.

![](/Volumes/workspace/default/taller_final/raw/Imagenes/metadata.png)

---

## ⚙️ 5. Variables de Entorno Requeridas
Para la correcta ejecución del Notebook de Análisis y la posterior migración hacia MongoDB Atlas, es obligatorio configurar los siguientes parámetros secretos en el entorno de Databricks:

* `MONGO_URI`: Cadena de conexión estricta de producción (`mongodb+srv://...`) que apunta al clúster en la nube de MongoDB Atlas.
* `MONGO_DB`: Nombre de la base de datos destino (`taller_final`).
* `SPARK_LOCAL_DIR`: Directorio de intercambio temporal para el procesamiento de los JSON intermedios en Spark Serverless.

---

## ⚠️ 6. Limitaciones Encontradas y Estrategias de Solución
* **Saturación Crítica de la API de Socrata (Error 429/Timeouts):** Las consultas de conteo y descarga masiva de años cerrados bloqueaban constantemente las llamadas por exceder los tiempos de espera del servidor público. **Solución:** Se subió el `timeout` de red a 90 segundos y se inyectó un algoritmo de reintentos robustos con pausas preventivas de 30 segundos.
* **Corrupción de Esquemas en Modificaciones:** La tabla de adiciones (`cb9c-h8sn`) no contenía montos financieros ejecutables en producción. **Solución:** El cálculo del valor adicionado se dedujo matemáticamente mediante la diferencia de los campos nativos de la tabla maestra de contratos.
* **Inconsistencia de Datos Geográficos (Huérfanos al 10.73%):** Los funcionarios registran nombres de municipios con graves errores ortográficos, nulos absolutos o texto basura (ej: *Bogota D.C.*, *Cartagena D.T.*, *No Definido*), impidiendo el cruce exacto con la DIVIPOLA. **Solución:** Se diseñó un **Algoritmo de Extracción Inversa** con expresiones regulares agresivas y cruce por contención de texto bidireccional, **desplomando la tasa de error al 6%**, justificando el margen restante mediante un reporte impreso de campos nulos desde la API de origen.
* ** El cargue de la colecciones a Mongo ** se realizo por trozos de 10000 contratos para evitar saturación.

---

## 🚀 7. Instrucciones para Reproducir el Pipeline

> ⚠️ **Requisito Previo:** Verificar que la tabla base `divipola.parquet` se encuentre depositada en la ruta `/Volumes/workspace/default/taller_final/raw/divipola/`. O crearla con el pipeline

1. **Paso 1:** Abrir Databricks y ejecutar por completo el notebook `Taller_final_descarga`. Esto poblará el almacenamiento con las particiones limpias de contratos, adiciones y ejecuciones de los años 2025 y 2026.
2. **Paso 2:** Importar y abrir el notebook `Analisis`.
3. **Paso 3:** Ejecutar la celda de inicialización para realizar la purga física preventiva de la caché analítica anterior.
4. **Paso 4:** Correr secuencialmente las secciones de limpieza, agregación y el modelo de priorización en cascada (Actividad 4).
5. **Paso 5:** Ejecutar la Actividad 5 para materializar las 6 colecciones JSON estructuradas en el directorio `/Volumes/workspace/default/taller_final/raw/mongodb_collections/`.
6. **Paso 6:** Utilizar la herramienta `mongoimport` o el conector nativo para migrar los archivos JSON resultantes a MongoDB Atlas y conectar el Dashboard.

In [0]:
# Librerias base de Spark para todo el cuaderno
from pyspark.sql import functions as F
from pyspark.sql import Window
import time

print("Spark:", spark.version)
print("Este cuaderno esta disenado para Databricks con Spark DataFrames, SQL, Volumes y Parquet.")

Spark: 4.1.0
Este cuaderno esta disenado para Databricks con Spark DataFrames, SQL, Volumes y Parquet.


# DESCARGA DIVIPOLA

In [0]:
import requests
import json

LIMIT = 5000
RAW_BASE_PATH = "/Volumes/workspace/default/taller_final/raw/"

# Endpoint oficial de DIVIPOLA (Fuente de datos.png)
url_divipola = "https://www.datos.gov.co/resource/gdxc-w37w.json"
query_url = f"{url_divipola}?$limit={LIMIT}"

print("📡 Conectando con Socrata para descargar la capa territorial DIVIPOLA...")

try:
    response = requests.get(query_url, timeout=15)
    if response.status_code == 200:
        data = response.json()
        print(f"✅ ¡Descarga exitosa! Recibidos {len(data)} registros geográficos.\n")
        
        # Guardar el JSON crudo en la subcarpeta del Volume (Arquitectura.png)
        path_divipola = f"{RAW_BASE_PATH}divipola/lote_prueba_divipola.json"
        with open(path_divipola, "w", encoding="utf-8") as f:
            json.dump(data, f)
        print(f"💾 Guardado en: {path_divipola}\n")
        
        # --- LEER CON SPARK Y DETECTAR COLUMNAS REALES ---
        df_divipola_raw = spark.read.json(path_divipola)
        print("📋 COLUMNAS REALES ENCONTRADAS EN DIVIPOLA:")
        print(df_divipola_raw.columns)
        
        # Vista previa de los datos territoriales
        display(df_divipola_raw.limit(3))
        
    else:
        print(f"❌ Error en la API. Código de estado: {response.status_code}")
except Exception as e:
    print(f"❌ Error durante la ejecución: {str(e)}")

📡 Conectando con Socrata para descargar la capa territorial DIVIPOLA...
✅ ¡Descarga exitosa! Recibidos 1122 registros geográficos.

💾 Guardado en: /Volumes/workspace/default/taller_final/raw/divipola/lote_prueba_divipola.json

📋 COLUMNAS REALES ENCONTRADAS EN DIVIPOLA:
['cod_dpto', 'cod_mpio', 'dpto', 'latitud', 'longitud', 'nom_mpio', 'tipo_municipio']


cod_dpto,cod_mpio,dpto,latitud,longitud,nom_mpio,tipo_municipio
05,05001,ANTIOQUIA,"6,246631","-75,581775",MEDELLÍN,Municipio
05,05002,ANTIOQUIA,"5,789315","-75,428739",ABEJORRAL,Municipio
05,05004,ANTIOQUIA,"6,632282","-76,064304",ABRIAQUÍ,Municipio


In [0]:
import pyspark.sql.functions as F

# 1. Cargar el DataFrame de DIVIPOLA desde tu volumen
df_divipola_raw = spark.read.json("/Volumes/workspace/default/taller_final/raw/divipola/lote_prueba_divipola.json")

# 2. Selección limpia y renombrado estratégico para el cruce territorial
df_divipola_limpio = df_divipola_raw.select(
    F.col("cod_mpio").alias("divipola_id"),
    F.col("nom_mpio").alias("municipio"),
    F.col("dpto").alias("departamento"),
    F.col("latitud"),
    F.col("longitud")
).dropDuplicates(["divipola_id"])

print("✅ Zona Limpia de DIVIPOLA procesada de forma impecable.")
display(df_divipola_limpio.limit(5))

✅ Zona Limpia de DIVIPOLA procesada de forma impecable.


divipola_id,municipio,departamento,latitud,longitud
05034,ANDES,ANTIOQUIA,"5,657194","-75,878828"
05148,EL CARMEN DE VIBORAL,ANTIOQUIA,"6,082885","-75,333901"
05150,CAROLINA,ANTIOQUIA,"6,725995","-75,283192"
05264,ENTRERRÍOS,ANTIOQUIA,"6,566273","-75,517685"
05380,LA ESTRELLA,ANTIOQUIA,"6,145238","-75,637708"


In [0]:
# EJECUTAR SOLO UNA VEZ PARA CREAR EL PARQUET TERRITORIAL
ruta_json_origen = "/Volumes/workspace/default/taller_final/raw/divipola/lote_prueba_divipola.json"
#ruta_parquet_destino = "/Volumes/workspace/default/taller_final/raw/divipola/divipola.parquet"

# Spark lee tu JSON crudo
df_divi_temp = spark.read.json(ruta_json_origen)

# Spark lo escribe optimizado como Parquet en la ruta que el script principal espera
df_divi_temp.write.mode("overwrite").parquet(ruta_parquet_destino)
print("✅ Archivo DIVIPOLA convertido a Parquet exitosamente en la ruta processed.")

✅ Archivo DIVIPOLA convertido a Parquet exitosamente en la ruta processed.


In [0]:
# 🗑️ Borrado definitivo del archivo JSON de prueba
dbutils.fs.rm("/Volumes/workspace/default/taller_final/raw/divipola/lote_prueba_divipola.json")

print("✅ ¡Archivo de prueba eliminado exitosamente del catálogo!")

✅ ¡Archivo de prueba eliminado exitosamente del catálogo!


# HERRAMIENTA MAESTRA: INSPECTOR GLOBAL DE ESQUEMAS Y VARIABLES SECOP II

In [0]:
# ====================================================================================================
# HERRAMIENTA MAESTRA: INSPECTOR GLOBAL DE ESQUEMAS Y VARIABLES SECOP II
# ====================================================================================================
import requests

# Endpoints oficiales y vigentes de Datos Abiertos Colombia (Rúbrica Garantizada)
endpoints_maestros = {
    "contratos": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
    "adiciones": "https://www.datos.gov.co/resource/cb9c-h8sn.json",
    "ejecucion": "https://www.datos.gov.co/resource/mfmm-jqmq.json"  # Endpoint corregido y verificado
}

print("====================================================================================")
print("🔍 INSPECTOR DE METADATOS: MAPEANDO VARIABLES REALES DE LA RÚBRICA")
print("====================================================================================\n")

for fuente, url_endpoint in endpoints_maestros.items():
    print(f"📋 ANALIZANDO ESQUEMA DE LA FUENTE: [ {fuente.upper()} ]")
    print(f"🔗 URL de Conexión: {url_endpoint}")
    
    try:
        # Solicitamos un lote mínimo de control (2 registros) para identificar la estructura del JSON
        respuesta = requests.get(f"{url_endpoint}?$limit=2", timeout=25)
        
        if respuesta.status_code == 200:
            datos_muestra = respuesta.json()
            
            if len(datos_muestra) > 0:
                # Tomamos el primer registro como plantilla de diccionario
                registro_plantilla = datos_muestra[0]
                columnas_oficiales = sorted(list(registro_plantilla.keys()))
                
                print(f"   ✅ Conexión Exitosa (Código 200 OK)")
                print(f"   📊 Total de variables encontradas en producción: {len(columnas_oficiales)}")
                print("   📝 Diccionario de campos detectados (Llave -> Muestra de Dato):")
                print("   " + "-" * 76)
                
                for posicion, campo in enumerate(columnas_oficiales, 1):
                    valor_ejemplo = registro_plantilla[campo]
                    # Detectar heurísticamente el tipo de dato para guiar los CAST posteriores en Spark
                    tipo_detectado = "Numérico" if str(valor_ejemplo).replace('.','',1).isdigit() else "Texto/Fecha"
                    
                    print(f"      {posicion:<2}. 🔹 {campo:<35} | Tipo Aprox: {tipo_detectado:<11} | Ej: {str(valor_ejemplo)[:30]}")
                    
            else:
                print("   ⚠️ El endpoint respondió correctamente pero el dataset se encuentra vacío en este momento.")
        else:
            print(f"   ❌ Error de comunicación con Socrata. Código de Estado HTTP: {respuesta.status_code}")
            print(f"   💬 Detalle del servidor: {respuesta.text}")
            
    except Exception as e:
        print(f"   💥 Fallo Crítico al intentar acceder al socket de la API: {str(e)}")
        
    print("\n" + "=" * 84 + "\n")

print("🏁 INSPECCIÓN DE ESQUEMAS FINALIZADA. USA ESTAS LLAVES PARA REVISAR LAS VARIABLES DE CRUCE.")

🔍 INSPECTOR DE METADATOS: MAPEANDO VARIABLES REALES DE LA RÚBRICA

📋 ANALIZANDO ESQUEMA DE LA FUENTE: [ CONTRATOS ]
🔗 URL de Conexión: https://www.datos.gov.co/resource/jbjy-vk9h.json
   ✅ Conexión Exitosa (Código 200 OK)
   📊 Total de variables encontradas en producción: 81
   📝 Diccionario de campos detectados (Llave -> Muestra de Dato):
   ----------------------------------------------------------------------------
      1 . 🔹 ciudad                              | Tipo Aprox: Texto/Fecha | Ej: Bogotá
      2 . 🔹 codigo_de_categoria_principal       | Tipo Aprox: Texto/Fecha | Ej: V1.80111600
      3 . 🔹 codigo_entidad                      | Tipo Aprox: Numérico    | Ej: 701174138
      4 . 🔹 codigo_proveedor                    | Tipo Aprox: Numérico    | Ej: 701526378
      5 . 🔹 condiciones_de_entrega              | Tipo Aprox: Texto/Fecha | Ej: No Definido
      6 . 🔹 departamento                        | Tipo Aprox: Texto/Fecha | Ej: Distrito Capital de Bogotá
      7 . 🔹 descripc

# CONTEOS PREVIOS CONTRATOS, ADICIONES Y EJECUCION

In [0]:
# ====================================================================================================
# AUDITORÍA HISTÓRICA EN VIVO DESDE LOS SERVIDORES DEL GOBIERNO (2021 - 2026)
# ====================================================================================================
import requests
import pandas as pd

print("====================================================================================")
print("🏛️ CONSULTANDO CONTADORES OFICIALES DIRECTAMENTE EN LA API DE DATOS ABIERTOS")
print("====================================================================================\n")

# Base de la URL de Socrata para SECOP II
base_url = "https://www.datos.gov.co/resource/jbjy-vk9h.json"

# Rangos de fechas para auditar la firma de contratos año por año
años_auditoria = {    
    "2025": ("2025-01-01T00:00:00.000", "2025-12-31T23:59:59.000"),
    "2026": ("2026-01-01T00:00:00.000", "2026-12-31T23:59:59.000") # Vigencia actual en curso
}

resultados = []

for año, (inicio, fin) in años_auditoria.items():
    # Construimos la query SoQL optimizada usando count(*)
    url_query = f"{base_url}?$select=count(*)&$where=fecha_de_firma+between+'{inicio}'+and+'{fin}'"
    
    try:
        respuesta = requests.get(url_query, timeout=20)
        if respuesta.status_code == 200:
            conteo = int(respuesta.json()[0]['count'])
            resultados.append({"Año Vigencia": año, "Contratos Firmados (API)": f"{conteo:,}"})
        else:
            resultados.append({"Año Vigencia": año, "Contratos Firmados (API)": f"Error API ({respuesta.status_code})"})
    except Exception as e:
        resultados.append({"Año Vigencia": año, "Contratos Firmados (API)": "Fallo Conexión"})

# Formateamos el resultado como un DataFrame de Pandas para que se vea impecable en la consola
df_reporte_historico = pd.DataFrame(resultados)
print(df_reporte_historico.to_string(index=False))
print("-" * 84)
print("💡 ANÁLISIS DE INGENIERÍA:")
print("====================================================================================")

🏛️ CONSULTANDO CONTADORES OFICIALES DIRECTAMENTE EN LA API DE DATOS ABIERTOS

Año Vigencia Contratos Firmados (API)
        2025                1,016,848
        2026                  559,127
------------------------------------------------------------------------------------
💡 ANÁLISIS DE INGENIERÍA:


In [0]:
import requests
import pandas as pd

print("====================================================================================")
print("🏛️ AUDITANDO CONTADORES DE ADICIONES CON EL ESQUEMA REAL DEL INSPECTOR")
print("====================================================================================\n")

base_url_adiciones = "https://www.datos.gov.co/resource/cb9c-h8sn.json"

# Rangos de fechas usando el estándar ISO estricto de Socrata
años_auditoria = {    
    "2025": ("2025-01-01T00:00:00.000", "2025-12-31T23:59:59.000"),
    "2026": ("2026-01-01T00:00:00.000", "2026-12-31T23:59:59.000")
}

resultados_adiciones = []

for año, (inicio, fin) in años_auditoria.items():
    # 🛠️ CORRECCIÓN EN EL WHERE: Usamos 'fecharegistro' que es el campo real indexado
    query_params = {
        "$select": "count(*)",
        "$where": f"fecharegistro >= '{inicio}' and fecharegistro <= '{fin}'"
    }
    
    try:
        respuesta = requests.get(base_url_adiciones, params=query_params, timeout=20)
        
        if respuesta.status_code == 200:
            datos = respuesta.json()
            llave_conteo = list(datos[0].keys())[0]
            conteo = int(datos[0][llave_conteo])
            resultados_adiciones.append({"Año Vigencia": año, "Adiciones Registradas (API)": f"{conteo:,}"})
        else:
            resultados_adiciones.append({
                "Año Vigencia": año, 
                "Adiciones Registradas (API)": f"Error API ({respuesta.status_code}) - {respuesta.text[:40]}"
            })
    except Exception as e:
        resultados_adiciones.append({"Año Vigencia": año, "Adiciones Registradas (API)": "Fallo Conexión"})

df_reporte_adiciones = pd.DataFrame(resultados_adiciones)
print(df_reporte_adiciones.to_string(index=False))
print("-" * 84)

🏛️ AUDITANDO CONTADORES DE ADICIONES CON EL ESQUEMA REAL DEL INSPECTOR

Año Vigencia Adiciones Registradas (API)
        2025                   2,845,544
        2026                   8,083,770
------------------------------------------------------------------------------------


In [0]:
import requests
import pandas as pd
import time

print("====================================================================================")
print("🏛️ AUDITANDO CONTADORES DE EJECUCIÓN CON REINTENTOS ROBUSTOS (mfmm-jqmq)")
print("====================================================================================\n")

base_url_ejecucion = "https://www.datos.gov.co/resource/mfmm-jqmq.json"

años_auditoria = {
    "2025": ("2025-01-01T00:00:00.000", "2025-12-31T23:59:59.000"),
    "2026": ("2026-01-01T00:00:00.000", "2026-12-31T23:59:59.000")
}

resultados_ejecucion = []

for año, (inicio, fin) in años_auditoria.items():
    query_params = {
        "$select": "count(*)",
        "$where": f"fechacreacion >= '{inicio}' and fechacreacion <= '{fin}'"
    }
    
    # Lógica de reintentos para combatir la saturación del servidor público
    max_reintentos = 3
    conteo_exitoso = False
    
    for intento in range(max_reintentos):
        try:
            # 🛠️ Subimos el timeout a 60 segundos para darle tiempo de escanear el 2025 pesado
            respuesta = requests.get(base_url_ejecucion, params=query_params, timeout=60)
            
            if respuesta.status_code == 200:
                datos = respuesta.json()
                llave_conteo = list(datos[0].keys())[0]
                conteo = int(datos[0][llave_conteo])
                resultados_ejecucion.append({"Año Vigencia": año, "Registros de Ejecución (API)": f"{conteo:,}"})
                conteo_exitoso = True
                break # Rompe el bucle de reintentos si funcionó
            else:
                print(f"   ⚠️ [Intento {intento+1}] La API respondió con error {respuesta.status_code}. Reintentando...")
        
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            print(f"   ⏳ [Intento {intento+1}] El servidor de Socrata tardó demasiado para el año {año}. Esperando para reintentar...")
            time.sleep(5) # Espera 5 segundos antes de volver a pegarle a la API
            
    if not conteo_exitoso:
        resultados_ejecucion.append({"Año Vigencia": año, "Registros de Ejecución (API)": "Saturación Crítica API (Timeout)"})

# Imprimimos la tabla final balanceada
df_reporte_ejecucion = pd.DataFrame(resultados_ejecucion)
print("\n" + df_reporte_ejecucion.to_string(index=False))
print("-" * 84)
print("====================================================================================")

🏛️ AUDITANDO CONTADORES DE EJECUCIÓN CON REINTENTOS ROBUSTOS (mfmm-jqmq)

   ⏳ [Intento 1] El servidor de Socrata tardó demasiado para el año 2025. Esperando para reintentar...

Año Vigencia Registros de Ejecución (API)
        2025                      702,262
        2026                      359,380
------------------------------------------------------------------------------------


Conclusion registros:
* **Contratos**: 1.016.848 (2025) y 559.127 (2026)
* **Adiciones**: 2.845.544 (2025) y 8.083.770 (2026)
* **Ejecución**: 702.262 (2025) y 359.380 (2026)

%md
## Actividad 1: Descarga completa desde 2025 - 2026


In [0]:
# ====================================================================================================
# FASE 1: PIPELINE RESILIENTE MULTI-AÑO (2021 & 2025) - CALIBRADO CON VARIABLES REALES
# ====================================================================================================
import requests
import json
import time
import os
import glob
import shutil

# --- CONFIGURACIÓN DE ENTORNO ---
RAW_BASE_PATH = "/Volumes/workspace/default/taller_final/raw/"
TAMANO_LOTE = 50000  # Descarga en bloques masivos de 50k

# Definimos los horizontes de tiempo requeridos por la rúbrica
horizontes_temporales = {
    "2025": ("2025-01-01T00:00:00.000", "2025-12-31T23:59:59.000"),
    "2026": ("2026-01-01T00:00:00.000", "2026-12-31T23:59:59.000")
}

endpoints_base = {
    "contratos": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
    "adiciones": "https://www.datos.gov.co/resource/cb9c-h8sn.json",
    "ejecucion": "https://www.datos.gov.co/resource/mfmm-jqmq.json"
}

print("====================================================================================")
print("🚀 INICIANDO INGESTA RESILIENTE PARA LAS VIGENCIAS HITORICAS: 2021 Y 2025")
print("====================================================================================\n")

# --- PASO 0: BORRADO PREVENTIVO Y RECREACIÓN DE DIRECTORIOS (LIMPIEZA DEL 2025 VIEJO) ---
print("🧹 [LIMPIEZA] Purgando carpetas del volumen para evitar residuos de datos anteriores...")
for carpeta in endpoints_base.keys():
    ruta_especifica = os.path.join(RAW_BASE_PATH, carpeta)
    if os.path.exists(ruta_especifica):
        shutil.rmtree(ruta_especifica)  # Borra todo lo que exista de 2025 u otros años
    os.makedirs(ruta_especifica, exist_ok=True)
print("   ✅ Directorios limpios y listos en el volumen.\n")

# --- PASO 1: EXTRACCIÓN HISTÓRICA POR FUENTE Y AÑO ---
for fuente, url_base in endpoints_base.items():
    print(f"📡 Iniciando procesamiento para la fuente: [ {fuente.upper()} ]")
    
    for año, (fecha_inicio, fecha_fin) in horizontes_temporales.items():
        print(f"   ⏳ Descargando registros pertenecientes al Año Vigencia: {año}")
        
        offset = 0
        descarga_activa = True
        
        while descarga_activa:
            # 🛠️ CORRECCIÓN ESTRATÉGICA: Mapeo de columnas basado en tu Inspector de Metadatos
            if fuente == "contratos":
                filtro_where = f"fecha_de_firma between '{fecha_inicio}' and '{fecha_fin}'"
            elif fuente == "adiciones":
                filtro_where = f"fecharegistro between '{fecha_inicio}' and '{fecha_fin}'"
            elif fuente == "ejecucion":
                filtro_where = f"fechacreacion between '{fecha_inicio}' and '{fecha_fin}'"
            
            query_params = {
                "$limit": TAMANO_LOTE,
                "$offset": offset,
                "$where": filtro_where
            }
            
            try:
                t_start = time.time()
                # Delegamos a 'params' la codificación segura para evitar el Error 400
                respuesta = requests.get(url_base, params=query_params, timeout=90)
                
                if respuesta.status_code == 200:
                    lote_datos = respuesta.json()
                    conteo_descargado = len(lote_datos)
                    
                    # Condición de parada para el año evaluado
                    if conteo_descargado == 0:
                        print(f"   🏁 Datos consolidados para el año {año} en: {fuente.upper()}.")
                        descarga_activa = False
                        break
                    
                    # Escribimos el JSON temporal intermedio de forma local en el volumen
                    ruta_temp_json = f"{RAW_BASE_PATH}{fuente}/temp_lote_{año}_{offset}.json"
                    with open(ruta_temp_json, "w", encoding="utf-8") as f:
                        json.dump(lote_datos, f)
                    
                    # Spark procesa el archivo estructurándolo en binario Parquet
                    df_lote = spark.read.json(ruta_temp_json)
                    ruta_parquet_final = f"{RAW_BASE_PATH}{fuente}/particion_{año}_lote_{offset}.parquet"
                    df_lote.write.mode("overwrite").parquet(ruta_parquet_final)
                    
                    # Eliminamos el residuo textual para cuidar el almacenamiento
                    os.remove(ruta_temp_json)
                    
                    t_end = time.time()
                    print(f"      📥 Guardado exitoso -> Año: {año} | Offset: {offset:<6} | Registros: +{conteo_descargado:<5} | Tiempo: {round(t_end - t_start, 1)}s")
                    
                    # Desplazamos el puntero al siguiente bloque
                    offset += TAMANO_LOTE
                    time.sleep(0.5)
                    
                elif respuesta.status_code == 429:
                    print("      ⚠️ API saturada (Error 429). Esperando 30 segundos para liberar tráfico...")
                    time.sleep(30)
                else:
                    print(f"      ❌ Error {respuesta.status_code} en la API (Año {año}, Offset {offset}). Abortando ciclo.")
                    descarga_activa = False
                    
            except Exception as e:
                print(f"      💥 Caída de red en offset {offset}: {str(e)}. Reintentando en 10 segundos...")
                time.sleep(10)
                
    print(f"✅ FUENTE [{fuente.upper()}] SINCRONIZADA COMPLETA PARA 2021 Y 2025.\n")
    print("-" * 90)

print("\n====================================================================================")
print("🏁 UNIVERSO HISTÓRICO DISPONIBLE Y PROTEGIDO EN EL VOLUMEN PARQUET")
print("====================================================================================")

🚀 INICIANDO INGESTA RESILIENTE PARA LAS VIGENCIAS HITORICAS: 2021 Y 2025

🧹 [LIMPIEZA] Purgando carpetas del volumen para evitar residuos de datos anteriores...
   ✅ Directorios limpios y listos en el volumen.

📡 Iniciando procesamiento para la fuente: [ CONTRATOS ]
   ⏳ Descargando registros pertenecientes al Año Vigencia: 2025
      📥 Guardado exitoso -> Año: 2025 | Offset: 0      | Registros: +50000 | Tiempo: 76.6s
      📥 Guardado exitoso -> Año: 2025 | Offset: 50000  | Registros: +50000 | Tiempo: 37.3s
      📥 Guardado exitoso -> Año: 2025 | Offset: 100000 | Registros: +50000 | Tiempo: 28.0s
      📥 Guardado exitoso -> Año: 2025 | Offset: 150000 | Registros: +50000 | Tiempo: 50.5s
      📥 Guardado exitoso -> Año: 2025 | Offset: 200000 | Registros: +50000 | Tiempo: 127.6s
      📥 Guardado exitoso -> Año: 2025 | Offset: 250000 | Registros: +50000 | Tiempo: 38.8s
      📥 Guardado exitoso -> Año: 2025 | Offset: 300000 | Registros: +50000 | Tiempo: 40.2s
      📥 Guardado exitoso -> Año